# 03. Feature Engineering

**Mục tiêu**: Build master dataset từ 14 bảng đã processed.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.feature_engineering import (
    build_master_dataset,
    add_temporal_features,
    add_route_historical_delay,
    aggregate_safety_features,
    add_truck_maintenance_features,
)
from src.preprocessing import time_based_split

PROC_DIR = Path('../data_processed')
OUT_DIR = Path('../data_features')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
tables = {p.stem: pd.read_parquet(p) for p in PROC_DIR.glob('*.parquet')}
print('Loaded', list(tables.keys()))

## 1. Build master dataset

In [ ]:
df = build_master_dataset(tables)
print('Master shape:', df.shape)
df.head(3)

## 2. Compute target: delay_hours

In [ ]:
events = tables['delivery_events']
delivery = events[events['event_type'] == 'Delivery'].copy()
delivery['delay_hours'] = (
    delivery['actual_datetime'] - delivery['scheduled_datetime']
).dt.total_seconds() / 3600

target = delivery.groupby('trip_id')['delay_hours'].mean().reset_index()
df = df.merge(target, on='trip_id', how='left').dropna(subset=['delay_hours'])
print('After target merge:', df.shape)

## 3. Temporal features

In [ ]:
df = add_temporal_features(df, 'dispatch_date')
df.head(2)

## 4. Historical features (no leakage)

In [ ]:
df = add_route_historical_delay(df)
df = add_truck_maintenance_features(df, tables['maintenance_records'])
df = aggregate_safety_features(df, tables['safety_incidents'])

## 5. Time-based split 70/15/15

In [ ]:
train, val, test = time_based_split(df, 'dispatch_date')
print('Train:', train.shape, 'Val:', val.shape, 'Test:', test.shape)

train.to_parquet(OUT_DIR / 'train.parquet', index=False)
val.to_parquet(OUT_DIR / 'val.parquet', index=False)
test.to_parquet(OUT_DIR / 'test.parquet', index=False)
print('Saved.')